In [1]:
import numpy as np
import pandas as pd

# Dataset & DataLoader

In [2]:
import torch
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as T

from dataset import TrainDataset, TestDataset

image_size = 64
batch_size = 128
mean = (0.485, 0.456, 0.406)
std  = (0.229, 0.224, 0.225)

train_transform = T.Compose([
    T.RandomResizedCrop(image_size, scale=(0.9, 1.0), ratio=(0.9, 1.1)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(20),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    T.RandomGrayscale(p=0.1),
    T.ToTensor(),
    T.Normalize(mean, std),
])

eval_transform = T.Compose([
    T.Resize((64, 64)),
    T.ToTensor(),
    T.Normalize(mean, std),
])

train_dataset = TrainDataset(root_path = './cs441-assn3-data/Train_64/', transform = train_transform)
test_dataset = TestDataset(root_path = './cs441-assn3-data/Test_64/', transform = eval_transform)

val_ratio = 0.2 # train 80%, val 20%

# 전체 길이 기준으로 train/val 길이 계산
n_total = len(train_dataset)
n_val = int(n_total * val_ratio)
n_train = n_total - n_val

train_dataset_split, val_dataset = random_split(
    train_dataset,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(dataset=train_dataset_split,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    drop_last = True,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

test_loader = DataLoader(dataset=test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4
)

# Your Awesome Model

In [3]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())


2.8.0+cu129
True


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# 1. Swish Activation (SiLU)
class Swish(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)

# 2. Squeeze-and-Excitation (SE) Block
class SEBlock(nn.Module):
    def __init__(self, in_channels, reduced_dim):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, reduced_dim, 1),
            nn.SiLU(),
            nn.Conv2d(reduced_dim, in_channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.se(x)

# 3. MBConv Block (Inverted Residual Block)
class MBConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, expand_ratio, reduction=4):
        super().__init__()
        self.use_residual = (in_channels == out_channels) and (stride == 1)
        hidden_dim = in_channels * expand_ratio
        
        layers = []
        # Expansion Phase (1x1 Conv)
        if expand_ratio != 1:
            layers.append(nn.Sequential(
                nn.Conv2d(in_channels, hidden_dim, 1, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.SiLU()
            ))

        # Depthwise Conv
        layers.append(nn.Sequential(
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size, stride, 
                      padding=kernel_size//2, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.SiLU()
        ))

        # Squeeze-and-Excitation
        reduced_dim = max(1, in_channels // reduction)
        layers.append(SEBlock(hidden_dim, reduced_dim))

        # Pointwise Conv (Project back)
        layers.append(nn.Sequential(
            nn.Conv2d(hidden_dim, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        ))

        self.block = nn.Sequential(*layers)

    def forward(self, x):
        res = self.block(x)
        if self.use_residual:
            return x + res
        return res

# 4. EfficientNet Main Class
class EfficientNet(nn.Module):
    def __init__(self, num_classes=15, width_mult=1.0, depth_mult=1.0):
        super().__init__()
        
        # EfficientNet-B0 Configuration
        # (in_channels, out_channels, kernel_size, stride, expand_ratio, repeats)
        base_settings = [
            [32, 16, 3, 1, 1, 1],
            [16, 24, 3, 2, 6, 2],  # Stride 2 -> 32x32
            [24, 40, 5, 2, 6, 2],  # Stride 2 -> 16x16
            [40, 80, 3, 2, 6, 3],  # Stride 2 -> 8x8
            [80, 112, 5, 1, 6, 3], 
            [112, 192, 5, 2, 6, 4], # Stride 2 -> 4x4
            [192, 320, 3, 1, 6, 1],
        ]

        # Helper function to scale width
        def scale_width(w):
            return int(math.ceil(w * width_mult / 8) * 8)
        
        # Helper function to scale depth
        def scale_depth(d):
            return int(math.ceil(d * depth_mult))

        # Stem (Initial Layer)
        out_channels = scale_width(32)
        # 64x64 이미지 손실을 줄이기 위해 stride=1로 시작하거나, stride=2 유지 (B0 표준은 2)
        # 여기서는 B0 표준(stride=2)을 따르되, 64입력이므로 32부터 시작됨.
        # 정보 손실이 걱정되면 stride=1로 바꿔도 됩니다.
        self.stem = nn.Sequential(
            nn.Conv2d(3, out_channels, 3, stride=1, padding=1, bias=False), # stride 2->1로 변경 (64px 최적화)
            nn.BatchNorm2d(out_channels),
            nn.SiLU()
        )
        
        # Build Blocks
        layers = []
        in_channels = out_channels
        
        for input_c, output_c, k, s, expand, repeats in base_settings:
            out_c = scale_width(output_c)
            num_repeats = scale_depth(repeats)
            
            for i in range(num_repeats):
                stride = s if i == 0 else 1
                layers.append(MBConvBlock(in_channels, out_c, k, stride, expand))
                in_channels = out_c
                
        self.blocks = nn.Sequential(*layers)

        # Head
        last_channels = scale_width(1280)
        self.head = nn.Sequential(
            nn.Conv2d(in_channels, last_channels, 1, bias=False),
            nn.BatchNorm2d(last_channels),
            nn.SiLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.2),
            nn.Linear(last_channels, num_classes)
        )

        # Initialize weights
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.head(x)
        return x

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 15
model = EfficientNet(num_classes=15, width_mult=2.4, depth_mult=1.4).to(device)

# 파라미터 수 확인
num_params = sum(p.numel() for p in model.parameters())
print(f"EfficientNet Parameters: {num_params/1e6:.2f} M")

EfficientNet Parameters: 41.86 M


# Model parameter checking

In [7]:
# model parameters checking
num_params = sum(p.numel() for p in model.parameters())
print('The number of your model parameters :', num_params)
print('Parameter usage : ' + str(num_params/1000000) + '%')
if num_params > 100000000:
  raise Exception('Compress your model.')

The number of your model parameters : 41859163
Parameter usage : 41.859163%


# Model training

In [8]:
print("GPU count:", torch.cuda.device_count())
print("Current device index:", torch.cuda.current_device())
print("Current device name:", torch.cuda.get_device_name(torch.cuda.current_device()))

GPU count: 1
Current device index: 0
Current device name: NVIDIA GeForce RTX 4080 SUPER


In [9]:
import tqdm
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast

# 0. 벤치마크 켜기 (속도 향상)
torch.backends.cudnn.benchmark = True

scaler = torch.amp.GradScaler('cuda')

epochs = 15
save_path = "best_model.pth"
best_val_loss = float("inf")

criterion = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

for epoch in range(epochs):
    # TRAIN
    model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for x, y in tqdm.tqdm(train_loader, desc=f"Epoch {epoch} [Train]"):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            output = model(x)
            loss = criterion(output, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # 통계 (AMP로 계산된 output을 그대로 사용)
        train_loss_sum += loss.item() * y.size(0)
        
        # 예측값 계산 (여기는 그라디언트 필요 없으므로 detach 추천)
        preds = torch.argmax(output.detach(), dim=1)
        train_correct += (preds == y).sum().item()
        train_total += y.size(0)

    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total

    # VALIDATION (그대로 유지)
    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for x, y in tqdm.tqdm(val_loader, desc=f"Epoch {epoch} [Val]"):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            # 검증 때는 AMP를 굳이 안 써도 되지만, 쓰면 조금 더 빠를 수 있음
            with autocast():
                output = model(x)
                val_loss_batch = criterion(output, y)

            val_loss_sum += val_loss_batch.item() * y.size(0)
            preds = torch.argmax(output, dim=1)
            val_correct += (preds == y).sum().item()
            val_total += y.size(0)

    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )

    scheduler.step()

    # MODEL SAVE
    save_dict = {
        "epoch": epoch,
        "model_state_dict": (
            model.module.state_dict() if isinstance(model, nn.DataParallel)
            else model.state_dict()
        ),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_loss": val_loss,
        "val_acc": val_acc,
    }

    torch.save(save_dict, f'epoch_{epoch}.pth')

    # BEST MODEL SPECIFICATION
    if val_loss < best_val_loss:
        best_val_loss = val_loss

        save_dict = {
            "epoch": epoch,
            "model_state_dict": (
                model.module.state_dict() if isinstance(model, nn.DataParallel)
                else model.state_dict()
            ),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": val_loss,
            "val_acc": val_acc,
        }

        torch.save(save_dict, save_path)
        print(f"Best model saved at epoch {epoch} (val_loss={val_loss:.4f})")

Epoch 0 [Val]:   0%|          | 0/71 [00:00<?, ?it/s]C:\Users\MAIN\AppData\Local\Temp\ipykernel_18532\4046747907.py:64: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 0 [Val]: 100%|██████████| 71/71 [00:14<00:00,  5.06it/s]


Epoch 00 | Train Loss: 2.5523 | Train Acc: 0.1612 | Val Loss: 2.4077 | Val Acc: 0.2553
Best model saved at epoch 0 (val_loss=2.4077)


Epoch 1 [Val]: 100%|██████████| 71/71 [00:12<00:00,  5.81it/s]


Epoch 01 | Train Loss: 2.1646 | Train Acc: 0.2980 | Val Loss: 2.0008 | Val Acc: 0.3548
Best model saved at epoch 1 (val_loss=2.0008)


Epoch 2 [Val]: 100%|██████████| 71/71 [00:12<00:00,  5.85it/s]


...

Epoch 12 | Train Loss: 0.7663 | Train Acc: 0.7445 | Val Loss: 1.0465 | Val Acc: 0.6694
Best model saved at epoch 12 (val_loss=1.0465)


Epoch 13 [Val]: 100%|██████████| 71/71 [00:12<00:00,  5.85it/s]


Epoch 13 | Train Loss: 0.7111 | Train Acc: 0.7649 | Val Loss: 1.0471 | Val Acc: 0.6778


Epoch 14 [Val]: 100%|██████████| 71/71 [00:11<00:00,  5.97it/s]


Epoch 14 | Train Loss: 0.6830 | Train Acc: 0.7725 | Val Loss: 1.0463 | Val Acc: 0.6753
Best model saved at epoch 14 (val_loss=1.0463)


In [10]:
'''
# 모델 불러오기
checkpoint = torch.load("best_model.pth", map_location=device)

# 먼저 순수 모델을 만들고 로드
base_model = ConvNeXtBN(num_classes=15)
base_model.load_state_dict(checkpoint["model_state_dict"])

# 그 다음에 DataParallel로 감쌈
if torch.cuda.device_count() > 1:
    model = torch.nn.DataParallel(base_model)
else:
    model = base_model

model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
'''

'\n# 모델 불러오기\ncheckpoint = torch.load("best_model.pth", map_location=device)\n\n# 먼저 순수 모델을 만들고 로드\nbase_model = ConvNeXtBN(num_classes=15)\nbase_model.load_state_dict(checkpoint["model_state_dict"])\n\n# 그 다음에 DataParallel로 감쌈\nif torch.cuda.device_count() > 1:\n    model = torch.nn.DataParallel(base_model)\nelse:\n    model = base_model\n\nmodel = model.to(device)\n\noptimizer = torch.optim.Adam(model.parameters(), lr=0.001)\noptimizer.load_state_dict(checkpoint["optimizer_state_dict"])\n'

In [11]:
'''
# 모델 로드 검증
missing_keys, unexpected_keys = base_model.load_state_dict(
    checkpoint["model_state_dict"], strict=False
)

print("Missing keys:", missing_keys)
print("Unexpected keys:", unexpected_keys)

# 파라미터 값 확인
with torch.no_grad():
    w = base_model.downsample_layers[0][0].weight

print("Sample weight stats:")
print("  mean:", w.mean().item())
print("  std :", w.std().item())
print("  min :", w.min().item())
print("  max :", w.max().item())

# forward 테스트
model.eval()
with torch.no_grad():
    dummy = torch.randn(2, 3, 224, 224).to(device)
    out = model(dummy)

print("Output shape:", out.shape)
print("Has NaN:", torch.isnan(out).any().item())
'''

'\n# 모델 로드 검증\nmissing_keys, unexpected_keys = base_model.load_state_dict(\n    checkpoint["model_state_dict"], strict=False\n)\n\nprint("Missing keys:", missing_keys)\nprint("Unexpected keys:", unexpected_keys)\n\n# 파라미터 값 확인\nwith torch.no_grad():\n    w = base_model.downsample_layers[0][0].weight\n\nprint("Sample weight stats:")\nprint("  mean:", w.mean().item())\nprint("  std :", w.std().item())\nprint("  min :", w.min().item())\nprint("  max :", w.max().item())\n\n# forward 테스트\nmodel.eval()\nwith torch.no_grad():\n    dummy = torch.randn(2, 3, 224, 224).to(device)\n    out = model(dummy)\n\nprint("Output shape:", out.shape)\nprint("Has NaN:", torch.isnan(out).any().item())\n'

In [12]:
'''
print("Checkpoint epoch:", checkpoint.get("epoch", "No epoch key"))
print("Val Loss at save time:", checkpoint.get("val_loss", "N/A"))
print("Val Acc at save time:", checkpoint.get("val_acc", "N/A"))
'''

'\nprint("Checkpoint epoch:", checkpoint.get("epoch", "No epoch key"))\nprint("Val Loss at save time:", checkpoint.get("val_loss", "N/A"))\nprint("Val Acc at save time:", checkpoint.get("val_acc", "N/A"))\n'

# Submit
Do not edit the submission code below.

In [13]:
submit = pd.read_csv('./cs441-assn3-data/Test_64.csv')

# model parameters checking
num_params = sum(p.numel() for p in model.parameters())
print('The number of your model parameters :', num_params)
print('Parameter usage : ' + str(num_params/1000000) + '%')
if num_params > 100000000:
  raise Exception('Compress your model.')

total_prediction = list()
model.eval()
with torch.no_grad():
    for x in tqdm.tqdm(test_loader):
        x = torch.FloatTensor(x).cuda()
        output = model(x)
        predict = torch.argmax(output,dim=1)
        total_prediction.extend(predict.cpu().numpy())
    submit['label'] = total_prediction
    submit.to_csv('submission.csv',index=False)

The number of your model parameters : 41859163
Parameter usage : 41.859163%


100%|██████████| 59/59 [00:22<00:00,  2.62it/s]
